# Notebook 03 — Functions, Modules, and Packages

**Module:** 01 — Python & Scientific Computing  
**Notebook:** 03 of 20  
**Prerequisites:** Notebook 02  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Every function you write in this module will eventually live inside `compbio_utils`.
The difference between a script that works once and a package that works reliably is
function design: clear signatures, validated inputs, documented contracts, and
sensible defaults.

This notebook establishes the patterns used everywhere in `compbio_utils`.

---
## Step 2 — Intuition

A function is a contract: the caller promises to supply inputs of a specified shape;
the function promises to return outputs of a specified shape or to raise a named
exception for invalid inputs. A good biological function is unambiguous about
what 'GC content of an empty sequence' means — it doesn't silently return 0.0.

A module is a namespace. A package is a directory of modules with an `__init__.py`
that decides what the package *advertises* to the outside world.

---
## Step 3 — Biological Background

Nucleotide sequences are the primary data type in computational biology.

**Key operations on DNA sequences:**

| Operation | Definition | Biological significance |
|-----------|------------|------------------------|
| GC content | (G+C)/(A+T+G+C) | Correlates with thermostability; taxonomic signal |
| Complement | A↔T, G↔C | Base-pairing rules (Watson-Crick) |
| Reverse complement | Complement then reverse | 5'→3' orientation of the antisense strand |
| Translation | DNA codon → amino acid | The genetic code (codon table) |

These four functions are the first public API of `compbio_utils.sequence`.

---
## Step 4 — Mathematical Explanation

$$\text{GC content} = \frac{|\{i : s_i \in \{G,C\}\}|}{|s|}$$

where $s$ is a sequence of length $|s|$ and $s_i$ is the nucleotide at position $i$.

**Complement mapping:** Let $\sigma : \{A,T,G,C\} \to \{T,A,C,G\}$ be the Watson-Crick
complement function. Then the reverse complement of sequence $s = s_1 s_2 \ldots s_n$
is $\overline{s} = \sigma(s_n)\sigma(s_{n-1})\ldots\sigma(s_1)$.

---
## Step 5 — Computational Explanation

**Function design principles applied here:**

1. **Single responsibility** — each function does one thing.
2. **Fail loudly on bad input** — `ValueError` is better than silent garbage.
3. **Case normalization at the boundary** — normalise to uppercase inside the function,
   not in the caller.
4. **Type hints everywhere** — enables mypy, IDEs, and auto-documentation.
5. **Short docstring, not a novel** — one-line summary, then parameters/returns/raises.

**Module structure:**
```
compbio_utils/
├── __init__.py       ← imports gc_content, complement, reverse_complement from sequence
└── sequence.py       ← all nucleotide functions defined here
```

The `__init__.py` controls what `from compbio_utils import X` exposes.

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — gc_content (the final version for compbio_utils)
def gc_content(sequence: str) -> float:
    """
    Compute GC content as a fraction in [0.0, 1.0].

    Parameters
    ----------
    sequence : str
        DNA or RNA sequence string; case-insensitive.

    Returns
    -------
    float
        GC fraction in [0.0, 1.0].

    Raises
    ------
    ValueError
        If sequence is empty.
    TypeError
        If sequence is not a string.
    """
    if not isinstance(sequence, str):
        raise TypeError(f"Expected str, got {type(sequence).__name__}")
    seq = sequence.upper()
    if not seq:
        raise ValueError("Sequence must not be empty")
    return sum(1 for nt in seq if nt in "GC") / len(seq)

print(gc_content("ATGCGC"))      # 0.6667
print(gc_content("atgcgc"))      # same — case-insensitive
print(gc_content("AAAATT"))      # 0.0

In [ ]:
# Cell 6.2 — complement and reverse_complement
COMPLEMENT = str.maketrans("ACGTacgt", "TGCAtgca")

def complement(sequence: str) -> str:
    """
    Return the Watson-Crick complement (same 5'→3' orientation).

    Preserves case. Non-standard characters are left unchanged.
    """
    if not isinstance(sequence, str):
        raise TypeError(f"Expected str, got {type(sequence).__name__}")
    return sequence.translate(COMPLEMENT)

def reverse_complement(sequence: str) -> str:
    """
    Return the reverse complement (5'→3' of the antisense strand).
    """
    return complement(sequence)[::-1]

print(complement("ATGC"))           # TACG
print(reverse_complement("ATGC"))   # GCAT

# The reverse complement of a palindrome equals itself:
palindrome = "GAATTC"  # EcoRI recognition site
print(f"EcoRI site is palindrome: {reverse_complement(palindrome) == palindrome}")

In [ ]:
# Cell 6.3 — Moving functions to a module
# This cell writes the actual compbio_utils/sequence.py file
import pathlib

repo_root = pathlib.Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists(): break
    repo_root = repo_root.parent

seq_module = repo_root / "utilities" / "compbio_utils" / "sequence.py"

if seq_module.exists():
    print(f"sequence.py exists at: {seq_module}")
    print("Review and update with gc_content, complement, reverse_complement if not present.")
else:
    print(f"sequence.py not found at: {seq_module}")
    print("Exercise: create this file with the three functions defined above.")

In [ ]:
# Cell 6.4 — Demonstrating __all__ and module-level imports
# In compbio_utils/__init__.py, you would write:
#
# from .sequence import gc_content, complement, reverse_complement
# __all__ = ["gc_content", "complement", "reverse_complement"]
#
# This allows:
#   from compbio_utils import gc_content    # ← works because __init__.py re-exports it
#   import compbio_utils; compbio_utils.gc_content("ATGC")  # ← also works
#
# __all__ controls what 'from compbio_utils import *' exposes.
# The star-import usage is rare in scientific code but __all__ is still good practice.

print("Verify this pattern by importing from compbio_utils after completing the exercise:")
try:
    from compbio_utils import gc_content as cbu_gc
    print(f"  compbio_utils.gc_content('ATGC') = {cbu_gc('ATGC')}")
except ImportError:
    print("  compbio_utils not yet updated — complete the exercise first.")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Visual: GC content across a sliding window
import matplotlib.pyplot as plt

def sliding_gc(sequence: str, window: int = 10) -> list[float]:
    return [
        gc_content(sequence[i:i+window])
        for i in range(len(sequence) - window + 1)
    ]

# Synthetic sequence: GC-rich region in the middle
seq = "ATATATATAT" + "GCGCGCGCGC" * 3 + "ATATATATAT"
positions = list(range(len(seq) - 9))
gc_values = sliding_gc(seq, window=10)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(positions, gc_values, color="steelblue", linewidth=1.5)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="50% GC")
ax.set_xlabel("Position")
ax.set_ylabel("GC content")
ax.set_title("Sliding window GC content (window=10)")
ax.set_ylim(-0.05, 1.05)
ax.legend()
plt.tight_layout()
plt.show()
print(f"Sequence length: {len(seq)} bp — {len(gc_values)} windows")

---
## Step 8 — Exercises

1. Add `gc_content`, `complement`, and `reverse_complement` to
   `utilities/compbio_utils/sequence.py` and update `__init__.py` to re-export them.
2. Write `sliding_gc(sequence, window)` inside `sequence.py` with full type hints
   and docstring.
3. Add three `pytest` unit tests for each of the four functions (12 tests total).
   Edge cases to cover: empty string, single nucleotide, all-N sequence.
4. Why does `str.maketrans` + `translate` outperform a loop for complement?  
   Benchmark both approaches on a 100 kb synthetic sequence and report the speedup.

*Solutions go in `utilities/compbio_utils/sequence.py` and `tests/test_sequence.py`.*

---
## Quiz — Active Recall

1. What is the reverse complement of `5'-ATGCATGC-3'`? Write it without code.
2. Why does `complement` preserve case rather than uppercasing?
3. What is the difference between `__init__.py`'s job and `sequence.py`'s job?
4. What does `__all__` control? What happens without it?
5. The EcoRI recognition site is `GAATTC`. Why is `reverse_complement('GAATTC') == 'GAATTC'`
   biologically significant?

---
## Reflection

**Date completed:** ____________________

> *[Did all four functions pass their unit tests? Could you write reverse_complement from scratch with the paper closed? What gap, if any, did this notebook reveal?]*

---
*Next: `04_numpy_fundamentals.ipynb`*